## 1. DESCARGAR DATOS

In [1]:
from pathlib import Path
import openpyxl
import yfinance as yf

workspace_root = Path.cwd().resolve().parents[1]
outfile = workspace_root / "MSFT_historical.csv"

history = yf.download("MSFT", period="max", auto_adjust=False, progress=False)
if history.empty:
    raise RuntimeError("No se pudieron descargar datos históricos para MSFT")

history.to_csv(outfile)
print(f"CSV creado: {outfile}")
print(f"Filas descargadas: {len(history):,}")

CSV creado: /home/hacker-m/Desktop/UDD/2026/Trimestre_01/AI_Workshop/Group/iit414w-lab01-TheUltrakills/MSFT_historical.csv
Filas descargadas: 10,122


In [2]:
from pathlib import Path

import pandas as pd
import yfinance as yf

workspace_root = Path.cwd().resolve()
csv_candidates = [
    workspace_root / "MSFT_historical.csv",
    workspace_root / "EvalProyectos" / "Tarea 2" / "MSFT_historical.csv",
]
msft_csv = next((path for path in csv_candidates if path.exists()), None)
if msft_csv is None:
    raise FileNotFoundError("No se encontró MSFT_historical.csv ni en la raíz del workspace ni en la carpeta de la tarea")

msft_raw = pd.read_csv(msft_csv, header=[0, 1], index_col=0, parse_dates=True)
msft_prices = msft_raw["Adj Close"].squeeze("columns")

spy_history = yf.download(
    "SPY",
    start=msft_prices.index.min(),
    end=msft_prices.index.max() + pd.Timedelta(days=1),
    auto_adjust=False,
    progress=False,
)
if spy_history.empty:
    raise RuntimeError("No se pudieron descargar datos de mercado para SPY")

spy_prices = spy_history["Adj Close"].squeeze("columns")

returns = pd.concat(
    [
        msft_prices.pct_change().rename("MSFT"),
        spy_prices.pct_change().rename("SPY"),
    ],
    axis=1,
).dropna()

sigma_im = returns["MSFT"].cov(returns["SPY"])
sigma_m2 = returns["SPY"].var()
beta_i = sigma_im / sigma_m2

print(f"beta_i = sigma_im / sigma_m^2 = {beta_i:.4f}")
print("Frecuencia de los datos: diaria")
print(f"Largo de la muestra: {len(returns):,} observaciones de rendimientos")

beta_i = sigma_im / sigma_m^2 = 1.0978
Frecuencia de los datos: diaria
Largo de la muestra: 8,380 observaciones de rendimientos


### Respuesta 1

 El beta calculado es de 1.0978 Beta 5YL Daily

 La frecuencia de los datos fue diaria

 El largo de la muestra es de 8380 observaciones de rendimiento

 Esta empresa fue elegida debido a su gran tamaño y flujo en el mercado, lo que da por hecho que tendrán una gran cantidad de data historica para usar

Diferencia con YahooFin = 0.0078 Beta 5YL Monthly



ej3 : Tasa

In [6]:
rf_history = yf.download(
    "^IRX",
    start=msft_prices.index.min(),
    end=msft_prices.index.max() + pd.Timedelta(days=1),
    auto_adjust=False,
    progress=False,
)

if rf_history.empty:
    raise RuntimeError("No se pudieron descargar datos de ^IRX")

rf_df = rf_history[["Adj Close"]].rename(columns={"Adj Close": "RF_anual"})
rf_df["RF_diaria"] = (1 + rf_df["RF_anual"] / 100) ** (1 / 252) - 1

## EXPORTADO DE DATOS A EXCEL

In [ ]:

# Export MSFT and SPY data to Excel with multiple sheets
excel_file = workspace_root / "EvalProyectos" / "Tarea 2" / "MSFT_SPY_data.xlsx"

with pd.ExcelWriter(excel_file, engine="openpyxl") as writer:
    msft_raw.to_excel(writer, sheet_name="MSFT")
    spy_history.to_excel(writer, sheet_name="SPY")
    returns.to_excel(writer, sheet_name="Retornos")
    rf_df.to_excel(writer, sheet_name="RF")


print(f"Archivo Excel creado: {excel_file}")
print(f"Hojas: 'MSFT', 'SPY', 'Retornos' y '^IRX' ")


Archivo Excel creado: /home/hacker-m/Desktop/UDD/2026/Trimestre_01/AI_Workshop/Group/iit414w-lab01-TheUltrakills/EvalProyectos/Tarea 2/MSFT_SPY_data.xlsx
Hojas: 'MSFT' y 'SPY'
